# Telco Customer Churn Prediction
**Data Science and Machine Learning Portfolio - Draft**

**Author:** Xinyu Jia  
**Student ID:** C00304524  
**Dataset:** Telco Customer Churn (Kaggle, IBM simulated data)


In [ ]:
# ============================================================
# Telco Customer Churn Prediction
# Data Science and Machine Learning Portfolio - Draft
# Author: Xinyu Jia | Student ID: C00304524
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)

sns.set_style('whitegrid')
RANDOM_STATE = 42


In [ ]:
# Load the Telco dataset; a file picker opens automatically in Colab.
from google.colab import files
uploaded = files.upload()

csv_path = list(uploaded.keys())[0]
df = pd.read_csv(csv_path)

print('Dataset shape (rows, columns):', df.shape)
df.head()


In [ ]:
# Inspect data types, missing values and duplicates
df.info()
print('\nDuplicated rows:', df.duplicated().sum())
print('\nSummary statistics:')
df.describe(include='all')


In [ ]:
# Data cleaning
# 1. Drop the identifier column, which carries no predictive value.
df = df.drop('customerID', axis=1)

# 2. TotalCharges is loaded as text because of blank values; convert to numeric.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 3. Drop rows with missing TotalCharges (new customers with tenure = 0).
df = df.dropna().reset_index(drop=True)

print('Dataset shape after cleaning:', df.shape)
print('\nCustomer churn count:')
print(df['Churn'].value_counts())


In [ ]:
# EDA 1: overall churn distribution
plt.figure(figsize=(6, 5))
ax = sns.countplot(x='Churn', data=df)
plt.title('Customer Churn Distribution')
plt.xlabel('Churn (Yes = Left, No = Stayed)')
plt.ylabel('Customer Count')
for p in ax.patches:
    ax.annotate(int(p.get_height()),
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
plt.show()


In [ ]:
# EDA 2: churn by contract type
plt.figure(figsize=(8, 5))
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title('Customer Churn by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Customer Count')
plt.show()


In [ ]:
# EDA 3: monthly charges distribution by churn
plt.figure(figsize=(7, 5))
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges Distribution by Churn')
plt.xlabel('Churn')
plt.ylabel('Monthly Charges')
plt.show()


In [ ]:
# EDA 4: churn by payment method
plt.figure(figsize=(9, 5))
sns.countplot(x='PaymentMethod', hue='Churn', data=df)
plt.title('Customer Churn by Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Customer Count')
plt.xticks(rotation=15)
plt.show()


In [ ]:
# Feature engineering
# Encode the target: Yes -> 1, No -> 0
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop('Churn', axis=1)

# One-hot encode categorical features (avoids introducing fake ordering)
X = pd.get_dummies(X, drop_first=True)
print('Feature matrix shape after encoding:', X.shape)

# Stratified split keeps the churn ratio consistent in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print('Train set size:', X_train.shape)
print('Test set size:', X_test.shape)


In [ ]:
# EDA 5: correlation heatmap for numeric features
plt.figure(figsize=(10, 6))
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), cmap='coolwarm', annot=True, fmt='.2f')
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()


In [ ]:
# Model training: Random Forest classifier
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]


In [ ]:
# Model validation: key classification metrics
print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred), 4))
print('Recall   :', round(recall_score(y_test, y_pred), 4))
print('F1 score :', round(f1_score(y_test, y_pred), 4))
print('ROC-AUC  :', round(roc_auc_score(y_test, y_proba), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred,
      target_names=['Stayed (0)', 'Churned (1)']))


In [ ]:
# Confusion matrix with readable labels
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'])
plt.xlabel('Predicted Churn')
plt.ylabel('True Churn')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# ROC curve
plt.figure(figsize=(7, 6))
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_value = roc_auc_score(y_test, y_proba)
plt.plot(fpr, tpr, label='Random Forest (AUC = %.3f)' % auc_value)
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


In [ ]:
# Feature importance (top 15)
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)
plt.figure(figsize=(10, 8))
importances.tail(15).plot(kind='barh')
plt.title('Top 15 Feature Importances from Random Forest')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## Conclusion
1. The dataset is imbalanced: about 26.5% of customers churned.
2. Month-to-month contracts, electronic-check users and customers with higher monthly charges show clearly higher churn risk.
3. The Random Forest model predicts churn with the metrics reported above; the churn-class recall is the main limitation.
4. Business actions: promote longer contracts, move customers to automatic payment, and give retention offers to high-charge, short-tenure customers.
5. Future work: tune hyper-parameters with GridSearchCV, handle imbalance with class weighting or SMOTE, and adjust the decision threshold to improve recall.
